In [1]:
import os
os.chdir("/workspaces/data-engineering-zoomcamp/W7-streaming/src")
print(os.getcwd())

/workspaces/data-engineering-zoomcamp/W7-streaming/src


In [2]:
from models import Ride, ride_deserializer
from kafka import KafkaConsumer

In [3]:
server = 'kafka:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer
)

In [4]:
import psycopg2

conn = psycopg2.connect(
    host='postgres',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
Inserted 4100 rows...
Inserted 4200 rows...
Inserted 4300 rows...
Inserted 4400 r

In [1]:
from kafka import KafkaConsumer
from kafka.admin import KafkaAdminClient

BOOTSTRAP = "kafka:9092"   # use this from devcontainer
TOPIC = "rides"

try:
    # 1) Check broker + topic metadata
    admin = KafkaAdminClient(
        bootstrap_servers=BOOTSTRAP,
        request_timeout_ms=5000,
        api_version_auto_timeout_ms=5000
    )
    topics = admin.list_topics()
    print("Connected to Kafka.")
    print("Topics:", topics)

    if TOPIC not in topics:
        print(f"Topic '{TOPIC}' does NOT exist.")
    else:
        print(f"Topic '{TOPIC}' exists.")

        # 2) Try to read one message
        consumer = KafkaConsumer(
            TOPIC,
            bootstrap_servers=BOOTSTRAP,
            auto_offset_reset="earliest",
            enable_auto_commit=False,
            consumer_timeout_ms=5000,   # stop after 5s if no message
            value_deserializer=lambda v: v.decode("utf-8", errors="ignore")
        )

        msg_count = 0
        for msg in consumer:
            msg_count += 1
            print(f"\nFound message in topic '{TOPIC}':")
            print("partition:", msg.partition)
            print("offset   :", msg.offset)
            print("value    :", msg.value[:500])   # print first 500 chars
            break

        if msg_count == 0:
            print(f"Connected OK, topic '{TOPIC}' exists, but no messages were read within timeout.")

        consumer.close()

    admin.close()

except Exception as e:
    print("Kafka connection/topic check failed:")
    print(type(e).__name__, str(e))

Connected to Kafka.
Topics: ['rides', '__consumer_offsets']
Topic 'rides' exists.

Found message in topic 'rides':
partition: 0
offset   : 0
value    : {"PULocationID": 90, "DOLocationID": 100, "trip_distance": 0.85, "total_amount": 16.45, "tpep_pickup_datetime": 1761956471000}


In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    bootstrap_servers="host.docker.internal:29092",
    consumer_timeout_ms=5000,
)
print(consumer.bootstrap_connected())
print(consumer.topics())

True
{'rides'}
